# 00 – Chuẩn bị dữ liệu: chuyển CSV sang Parquet

**Mục đích.** Bộ MovieLens 20M gốc là file CSV (`rating.csv` khoảng 690 MB, 20 triệu dòng). Notebook này
chuyển CSV sang **Parquet một lần duy nhất**; các notebook 01–03 sau đó đọc từ Parquet.

**Vì sao Parquet nhanh hơn CSV?**
- CSV là văn bản thuần: mỗi lần đọc Spark phải tách chuỗi, đổi kiểu từng giá trị, và nếu dùng
  `inferSchema` còn phải quét thêm cả file để đoán kiểu.
- Parquet lưu sẵn **schema và kiểu dữ liệu**, lưu **theo cột** (chỉ đọc cột cần dùng) và **nén**
  nên file nhỏ hơn nhiều. Phần footer còn chứa thống kê (số dòng, min/max), nên một số phép như `count()`
  gần như không phải đọc dữ liệu.

**Ai chạy?** Mỗi thành viên tự chạy notebook này **một lần trên máy mình**, sau khi đã giải nén CSV vào
`data/ml-20m/` (xem `data/README.md`). Thư mục `data/` không đưa lên GitHub.

Kết quả tạo ra:
- `data/ml-20m-parquet/full/{rating,movie,tag}`: toàn bộ dữ liệu.
- `data/ml-20m-parquet/sample/{rating,movie,tag}`: 5% `rating` (seed 42), `movie` và `tag` giữ nguyên.

## 1. Khởi tạo SparkSession

Cell khởi tạo chung của nhóm (giống nhau ở mọi notebook). Chỉ được đổi `SIZE` và hai dòng cấu hình bộ nhớ.

In [1]:
# ANH: 00_01 | Khởi tạo SparkSession: phiên bản Spark và số luồng
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# "sample" khi viết code, đổi thành "full" khi chạy số liệu cuối
SIZE = "sample"
CSV  = "/home/jovyan/work/data/ml-20m"                    # CSV gốc (chỉ A dùng để so sánh)
DATA = f"/home/jovyan/work/data/ml-20m-parquet/{SIZE}"    # Parquet: B, C đọc từ đây

spark = (SparkSession.builder
         .appName("BigDataLab")
         .master("local[*]")                    # máy 8GB: "local[2]"
         .config("spark.driver.memory", "3g")   # máy 8GB: "2g"
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())
sc = spark.sparkContext

print("Spark version:", spark.version)
print("Số luồng song song:", sc.defaultParallelism)

Spark version: 4.2.0
Số luồng song song: 8


## 2. Xem nhanh dữ liệu CSV gốc

In kích thước từng file và vài dòng đầu (đọc trực tiếp bằng Python, không qua Spark) để xác nhận:
tên file số ít, có dòng tiêu đề, và cột `timestamp` là **chuỗi ngày giờ** (vd `2005-04-02 23:53:47`)
chứ không phải số giây — vì vậy dùng `F.to_timestamp`.

In [2]:
# ANH: 00_02 | Dữ liệu CSV gốc: kích thước và vài dòng đầu
import os

SRC = "/home/jovyan/work/data/ml-20m"
DST = "/home/jovyan/work/data/ml-20m-parquet"
TABLES = ["rating", "movie", "tag"]
MB = 1024 ** 2

for name in TABLES:
    path = f"{SRC}/{name}.csv"
    print(f"{name}.csv  ({os.path.getsize(path) / MB:,.1f} MB)")
    with open(path, encoding="utf-8") as f:
        for _ in range(4):                     # dòng tiêu đề + 3 dòng dữ liệu
            print("   ", f.readline().rstrip())
    print()

rating.csv  (658.4 MB)
    "userId","movieId","rating","timestamp"
    1,2,3.5,2005-04-02 23:53:47
    1,29,3.5,2005-04-02 23:31:16
    1,32,3.5,2005-04-02 23:33:39

movie.csv  (1.4 MB)
    "movieId","title","genres"
    1,"Toy Story (1995)","Adventure|Animation|Children|Comedy|Fantasy"
    2,"Jumanji (1995)","Adventure|Children|Fantasy"
    3,"Grumpier Old Men (1995)","Comedy|Romance"

tag.csv  (20.7 MB)
    "userId","movieId","tag","timestamp"
    18,4141,"Mark Waters",2009-04-24 18:19:40
    65,208,"dark hero",2013-05-10 01:41:18
    65,353,"dark hero",2013-05-10 01:41:19



## 3. Chuyển CSV sang Parquet (bản full)

- Khai báo **schema** sẵn thay vì `inferSchema`: Spark không phải quét file để đoán kiểu, đọc nhanh hơn và
  kiểu dữ liệu đúng như mong muốn (`userId`, `movieId` là `INT`, `rating` là `DOUBLE`).
- Cột `timestamp` đọc vào là chuỗi, rồi đổi sang kiểu `timestamp` bằng `F.to_timestamp`.
- Ghi bằng `mode("overwrite")` để chạy lại notebook không bị lỗi thư mục đã tồn tại.
- Đo thời gian ghi từng bảng. Vì Spark *lazy*, thời gian này gồm cả đọc CSV + đổi kiểu + ghi Parquet.

In [3]:
# ANH: 00_03 | Thời gian chuyển CSV sang Parquet (bản full)
# Khai báo schema để không phải đoán kiểu, đọc nhanh và đúng
rating = (spark.read.csv(f"{SRC}/rating.csv", header=True,
              schema="userId INT, movieId INT, rating DOUBLE, timestamp STRING")
          .withColumn("timestamp", F.to_timestamp("timestamp")))
movie  =  spark.read.csv(f"{SRC}/movie.csv", header=True,
              schema="movieId INT, title STRING, genres STRING")
tag    = (spark.read.csv(f"{SRC}/tag.csv", header=True,
              schema="userId INT, movieId INT, tag STRING, timestamp STRING")
          .withColumn("timestamp", F.to_timestamp("timestamp")))

write_time = {}
for name, df in {"rating": rating, "movie": movie, "tag": tag}.items():
    t0 = time.time()
    df.write.mode("overwrite").parquet(f"{DST}/full/{name}")
    write_time[name] = time.time() - t0
    print(f"{name:<7} CSV -> Parquet: {write_time[name]:7.2f} s")
print(f"{'Tổng':<7} {'':>16} {sum(write_time.values()):7.2f} s")

rating  CSV -> Parquet:   19.93 s


movie   CSV -> Parquet:    1.60 s


tag     CSV -> Parquet:    2.93 s
Tổng                       24.46 s


## 4. Tạo bản sample

Bản sample dùng khi viết và thử code cho nhanh: lấy ngẫu nhiên **5%** bảng `rating` với `seed=42`
(cả nhóm ra cùng một mẫu), còn `movie` và `tag` nhỏ nên giữ nguyên. Đọc từ Parquet full vừa tạo
(nhanh hơn đọc lại CSV).

In [4]:
# ANH: 00_04 | Tạo bản sample
t0 = time.time()
(spark.read.parquet(f"{DST}/full/rating")
      .sample(fraction=0.05, seed=42)
      .write.mode("overwrite").parquet(f"{DST}/sample/rating"))
for name in ["movie", "tag"]:
    df = spark.read.parquet(f"{DST}/full/{name}")
    df.write.mode("overwrite").parquet(f"{DST}/sample/{name}")
print(f"Tạo bản sample mất: {time.time() - t0:.2f} s")
print()

print(f"{'Bảng':<8}{'full':>14}{'sample':>14}{'tỉ lệ':>9}")
for name in TABLES:
    n_full   = spark.read.parquet(f"{DST}/full/{name}").count()
    n_sample = spark.read.parquet(f"{DST}/sample/{name}").count()
    print(f"{name:<8}{n_full:>14,}{n_sample:>14,}{n_sample / n_full:>9.2%}")

Tạo bản sample mất: 9.35 s

Bảng              full        sample    tỉ lệ


rating      20,000,263       999,837    5.00%


movie           27,278        27,278  100.00%


tag            465,564       465,564  100.00%


## 5. Kiểm tra chuyển đổi

### 5.1. Số dòng CSV và Parquet

Đếm lại số dòng từ CSV gốc (đọc không khai báo schema) và từ Parquet full. Hai số phải bằng nhau;
nếu lệch nghĩa là có dòng bị mất hoặc đọc sai khi chuyển đổi.

In [5]:
# ANH: 00_05 | Kiểm tra số dòng CSV và Parquet
print(f"{'Bảng':<8}{'CSV':>14}{'Parquet':>14}  Kết quả")
for name in TABLES:
    n_csv = spark.read.csv(f"{SRC}/{name}.csv", header=True).count()
    n_pq  = spark.read.parquet(f"{DST}/full/{name}").count()
    print(f"{name:<8}{n_csv:>14,}{n_pq:>14,}  {'OK' if n_csv == n_pq else 'LỆCH'}")

Bảng               CSV       Parquet  Kết quả


rating      20,000,263    20,000,263  OK


movie           27,278        27,278  OK


tag            465,564       465,564  OK


### 5.2. Schema bảng `rating` sau chuyển đổi

Cột `timestamp` phải có kiểu `timestamp` (không còn là `string`), và không có giá trị null
(null nghĩa là `to_timestamp` không đọc được chuỗi ngày giờ).

In [6]:
# ANH: 00_06 | Schema bảng rating sau chuyển đổi
r = spark.read.parquet(f"{DST}/full/rating")
r.printSchema()                                        # timestamp phải là kiểu timestamp
r.show(3)
n_null = r.filter(F.col("timestamp").isNull()).count()  # phải bằng 0
print("Số dòng timestamp null:", n_null, "(OK)" if n_null == 0 else "(CẦN KIỂM TRA)")

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: timestamp (nullable = true)



+------+-------+------+-------------------+
|userId|movieId|rating|          timestamp|
+------+-------+------+-------------------+
|     1|      2|   3.5|2005-04-02 23:53:47|
|     1|     29|   3.5|2005-04-02 23:31:16|
|     1|     32|   3.5|2005-04-02 23:33:39|
+------+-------+------+-------------------+
only showing top 3 rows


Số dòng timestamp null: 0 (OK)


## 6. So sánh CSV và Parquet

### 6.1. Dung lượng trên đĩa

Parquet là một **thư mục** gồm nhiều file `part-*.parquet` (Spark ghi song song), nên dung lượng được tính
bằng tổng kích thước mọi file trong thư mục. Tỉ lệ nén = dung lượng CSV / dung lượng Parquet full.

In [7]:
# ANH: 00_07 | So sánh dung lượng CSV và Parquet
def dir_size(path):
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return total

print(f"{'Bảng':<8}{'CSV (MB)':>11}{'Parquet full':>14}{'Tỉ lệ nén':>11}{'Parquet sample':>16}")
tot = [0, 0, 0]
for name in TABLES:
    s_csv  = os.path.getsize(f"{SRC}/{name}.csv")
    s_full = dir_size(f"{DST}/full/{name}")
    s_samp = dir_size(f"{DST}/sample/{name}")
    tot = [tot[0] + s_csv, tot[1] + s_full, tot[2] + s_samp]
    print(f"{name:<8}{s_csv / MB:>11,.1f}{s_full / MB:>14,.1f}{s_csv / s_full:>10.1f}x{s_samp / MB:>16,.1f}")
print(f"{'Tổng':<8}{tot[0] / MB:>11,.1f}{tot[1] / MB:>14,.1f}{tot[0] / tot[1]:>10.1f}x{tot[2] / MB:>16,.1f}")

Bảng       CSV (MB)  Parquet full  Tỉ lệ nén  Parquet sample
rating        658.4         171.2       3.8x            12.9
movie           1.4           0.7       2.0x             0.7


tag            20.7           6.6       3.1x             6.6
Tổng          680.5         178.5       3.8x            20.2


### 6.2. Thời gian đọc bảng `rating`

So sánh đọc + `count()` bảng `rating` (20 triệu dòng):
- **CSV với `inferSchema=True`**: cách đọc "mặc định" khi chưa có schema — Spark quét file một lượt để đoán
  kiểu, rồi `count()` quét thêm một lượt nữa.
- **Parquet**: schema có sẵn; `count()` lấy số dòng từ metadata trong footer của các file Parquet.

Lưu ý: thời gian phụ thuộc máy và việc file đã nằm trong cache của hệ điều hành hay chưa, nên chỉ dùng để so
sánh tương đối.

In [8]:
# ANH: 00_08 | Thời gian đọc CSV và Parquet
t0 = time.time()
df_csv = spark.read.csv(f"{SRC}/rating.csv", header=True, inferSchema=True)
n_csv = df_csv.count()
t_csv = time.time() - t0

t0 = time.time()
df_pq = spark.read.parquet(f"{DST}/full/rating")
n_pq = df_pq.count()
t_pq = time.time() - t0

print(f"{'Nguồn':<24}{'Số dòng':>14}{'Thời gian':>12}")
print(f"{'CSV (inferSchema)':<24}{n_csv:>14,}{t_csv:>10.2f} s")
print(f"{'Parquet':<24}{n_pq:>14,}{t_pq:>10.2f} s")
print(f"Parquet nhanh hơn khoảng {t_csv / t_pq:.0f} lần")

Nguồn                          Số dòng   Thời gian
CSV (inferSchema)           20,000,263     10.33 s
Parquet                     20,000,263      0.44 s
Parquet nhanh hơn khoảng 23 lần


## 7. Kết luận

Số liệu đo trên máy A (Docker: 8 CPU, 17,6 GiB RAM, `local[*]` = 8 luồng; thư mục dự án mount từ Windows):

- **Chuyển đổi đúng:** số dòng CSV và Parquet khớp ở cả 3 bảng (`rating` 20.000.263, `movie` 27.278,
  `tag` 465.564); cột `timestamp` có kiểu `timestamp`, không có giá trị null.
- **Thời gian chuyển đổi:** khoảng 24 s cho cả 3 bảng (riêng `rating` khoảng 20 s); tạo bản sample khoảng 9 s.
  Bản sample có 999.837 dòng `rating` (5,00%).
- **Dung lượng:** 680,5 MB CSV còn 178,5 MB Parquet (nhỏ hơn khoảng **3,8 lần**); bản sample chỉ 20,2 MB.
- **Tốc độ đọc:** đọc + `count()` bảng `rating` mất khoảng 10,3 s từ CSV (`inferSchema`) và chỉ khoảng 0,4 s từ
  Parquet, nhanh hơn khoảng **23 lần**.

→ Các notebook sau đọc dữ liệu từ Parquet (`DATA`), không đọc lại CSV. Thời gian có thể khác giữa các lần chạy
và giữa các máy; chỉ nên dùng để so sánh tương đối.

In [9]:
spark.stop()